# DSA Final Project, December 2025, Gavin Angeloff 

### Import Statements

In [33]:
import tensorflow as tf
import pandas as pd
from tensorflow import keras
import matplotlib as plt
# Can't directly import keras layers with from tensorflow.keras import layers
# because of dynamic import problems with library. Do not conflict with tensorflow.python.keras
# https://stackoverflow.com/questions/71000250/import-tensorflow-keras-could-not-be-resolved-after-upgrading-to-tensorflow-2
print(f"tensorflow_version: {tf.__version__}")
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))
# macOS 12.0 (Monterey) or higher (64-bit) (no GPU support)
# Windows Native - Windows 7 or higher (64-bit) (no GPU support after TF 2.10)
# TF is cooked with no GPU support. Why would they do that?


tensorflow_version: 2.20.0
Num GPUs Available:  0


### Dataset importing

In [34]:
print("ds_train:")
ds_train = tf.keras.preprocessing.image_dataset_from_directory(
    'card_data/train',
    labels='inferred',
    label_mode='int',
    class_names=None,
    color_mode='rgb',
    batch_size=32,
    image_size=(200, 200),
    shuffle=True,
    seed=None,
    validation_split=None,
    subset=None,
    interpolation='bilinear',
    follow_links=False,
    crop_to_aspect_ratio=False,
    pad_to_aspect_ratio=False,
    data_format=None,
    verbose=True
)
print("ds_valid:")
ds_valid = tf.keras.preprocessing.image_dataset_from_directory(
    'card_data/valid',
    labels='inferred',
    label_mode='int',
    class_names=None,
    color_mode='rgb',
    batch_size=32,
    image_size=(200, 200),
    shuffle=True,
    seed=None,
    validation_split=None,
    subset=None,
    interpolation='bilinear',
    follow_links=False,
    crop_to_aspect_ratio=False,
    pad_to_aspect_ratio=False,
    data_format=None,
    verbose=True
)

ds_train:
Found 7624 files belonging to 53 classes.
ds_valid:
Found 265 files belonging to 53 classes.


### Model

In [35]:
'''
model = keras.Sequential([
    keras.layers.Conv2D(filters=64, kernel_size=2, input_shape=[200,200,3]),
    keras.layers.GlobalAvgPool2D(),
    keras.layers.Dense(53, activation='softmax'),
])
'''

model = keras.Sequential([
    keras.layers.InputLayer([200,200,3]),

    # Data Augmentation
    # https://www.tensorflow.org/guide/keras/preprocessing_layers
    keras.layers.RandomCrop(200, 200),
    keras.layers.RandomFlip(),
    keras.layers.RandomTranslation(.5, .5),
    keras.layers.RandomRotation(0.5),
    keras.layers.RandomZoom(0.5),
    keras.layers.RandomContrast(0.5),

    # Block One
    keras.layers.BatchNormalization(),
    keras.layers.Conv2D(filters=64, kernel_size=3, activation='relu', padding='same'),
    keras.layers.MaxPool2D(),

    # Block Two
    keras.layers.BatchNormalization(),
    keras.layers.Conv2D(filters=128, kernel_size=3, activation='relu', padding='same'),
    keras.layers.MaxPool2D(),

    # Block Three
    keras.layers.BatchNormalization(),
    keras.layers.Conv2D(filters=256, kernel_size=3, activation='relu', padding='same'),
    keras.layers.Conv2D(filters=256, kernel_size=3, activation='relu', padding='same'),
    keras.layers.MaxPool2D(),

    # Head
    keras.layers.BatchNormalization(),
    keras.layers.GlobalAveragePooling2D(),
    keras.layers.Dense(53, activation='softmax'),
])

# Show Layers
model.summary()

Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ random_crop_7 (RandomCrop)      │ (None, 200, 200, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_flip_7 (RandomFlip)      │ (None, 200, 200, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_translation_7            │ (None, 200, 200, 3)    │             0 │
│ (RandomTranslation)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_rotation_7               │ (None, 200, 200, 3)    │             0 │
│ (RandomRotation)                │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_zoom_7 (RandomZoom)      │ (None, 200, 200, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_contrast_7               │ (None, 200, 200, 3)    │             0 │
│ (RandomContrast)                │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_19          │ (None, 200, 200, 3)    │            12 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_17 (Conv2D)              │ (None, 200, 200, 64)   │         1,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_12 (MaxPooling2D) │ (None, 100, 100, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_20          │ (None, 100, 100, 64)   │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_18 (Conv2D)              │ (None, 100, 100, 128)  │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_13 (MaxPooling2D) │ (None, 50, 50, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_21          │ (None, 50, 50, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_19 (Conv2D)              │ (None, 50, 50, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_20 (Conv2D)              │ (None, 50, 50, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_14 (MaxPooling2D) │ (None, 25, 25, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_22          │ (None, 25, 25, 256)    │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_3      │ (None, 256)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 53)             │        13,621 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 976,321 (3.72 MB)

 Trainable params: 975,419 (3.72 MB)

 Non-trainable params: 902 (3.52 KB)

In [36]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(epsilon=0.001),
    loss = 'sparse_categorical_crossentropy', # multiclass version of loss='binary_crossentropy',
    metrics=['sparse_categorical_accuracy'] # multiclass version of metrics=['binary_accuracy'],
)

history = model.fit(
    ds_train,
    validation_data=ds_valid,
    epochs=200,
)

Epoch 1/200
 11/239 ━━━━━━━━━━━━━━━━━━━━ 1:54 504ms/step - loss: 4.1497 - sparse_categorical_accuracy: 0.0337

KeyboardInterrupt: 

## Model Results

In [ ]:
# Plot learning curves
history_frame = pd.DataFrame(history.history)
history_frame.loc[:, ['loss', 'val_loss']].plot()
history_frame.loc[:, ['sparse_categorical_accuracy', 'val_sparse_categorical_accuracy']].plot()
plt.show()

## Save Trained Model


In [ ]:
# https://www.tensorflow.org/tutorials/keras/save_and_load#save_the_entire_model
# how to load
#new_model = tf.keras.models.load_model('model.keras')

# This saves:
# -The model's configuration (architecture)
# -The model's weights
# -The model's optimizer's state (if any)

model.save('trained_models/model200ep.keras')


## Sources:

Tensorflow
https://www.tensorflow.org/api_docs
